In [1]:
import numpy as np
import pandas as pd
import data_clean_utils
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer, MissingIndicator
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder, MinMaxScaler, PowerTransformer, OrdinalEncoder
from sklearn.model_selection import train_test_split

In [2]:
import dagshub
import mlflow
dagshub.init(repo_owner="rahulpatel16092005",repo_name="swiggy-delivery-time-prediction",mlflow=True)
mlflow.set_tracking_uri("https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow")

c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as rahulpatel16092005

Initialized MLflow to track repo "rahulpatel16092005/swiggy-delivery-time-prediction"

Repository rahulpatel16092005/swiggy-delivery-time-prediction initialized!

In [4]:
mlflow.set_experiment("Exp 3 - RF - HP Tuning")

2026/03/31 13:00:16 INFO mlflow.tracking.fluent: Experiment with name 'Exp 3 - RF - HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/df802e3198114a238e9d2d5b44713898', creation_time=1774942218178, experiment_id='4', last_update_time=1774942218178, lifecycle_stage='active', name='Exp 3 - RF - HP Tuning', tags={}, workspace='default'>

In [5]:
from sklearn import set_config

set_config(transform_output="pandas")

In [6]:
df=pd.read_csv("swiggy_cleaned.csv")

In [7]:
df.head()

,rider_id,age,ratings,restaurant_latitude,restaurant_longitude,delivery_latitude,delivery_longitude,order_date,weather,traffic,...,city_name,order_day,order_month,order_day_of_week,is_weekend,pickup_time_minutes,order_time_hour,order_time_of_day,distance,distance_type
0,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,2022-03-19,sunny,high,...,INDO,19,3,saturday,1,15.0,11.0,morning,3.025149,short
1,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,2022-03-25,stormy,jam,...,BANG,25,3,friday,0,5.0,19.0,evening,20.183530,very_long
2,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,2022-03-19,sandstorms,low,...,BANG,19,3,saturday,1,15.0,8.0,morning,1.552758,short
3,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,2022-04-05,sunny,medium,...,COIMB,5,4,tuesday,0,10.0,18.0,evening,7.790401,medium
4,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,2022-03-26,cloudy,high,...,CHEN,26,3,saturday,1,15.0,13.0,afternoon,6.210138,medium


In [8]:
# drop columns not required for model input

columns_to_drop =  ['rider_id',
                    'restaurant_latitude',
                    'restaurant_longitude',
                    'delivery_latitude',
                    'delivery_longitude',
                    'order_date',
                    "order_time_hour",
                    "order_day",
                    "city_name",
                    "order_day_of_week",
                    "order_month"]

df.drop(columns=columns_to_drop, inplace=True)

df

,age,ratings,weather,traffic,vehicle_condition,type_of_order,type_of_vehicle,multiple_deliveries,festival,city_type,time_taken,is_weekend,pickup_time_minutes,order_time_of_day,distance,distance_type
0,37.0,4.9,sunny,high,2,snack,motorcycle,0.0,no,urban,24,1,15.0,morning,3.025149,short
1,34.0,4.5,stormy,jam,2,snack,scooter,1.0,no,metropolitian,33,0,5.0,evening,20.183530,very_long
2,23.0,4.4,sandstorms,low,0,drinks,motorcycle,1.0,no,urban,26,1,15.0,morning,1.552758,short
3,38.0,4.7,sunny,medium,0,buffet,motorcycle,1.0,no,metropolitian,21,0,10.0,evening,7.790401,medium
4,32.0,4.6,cloudy,high,1,snack,scooter,1.0,no,metropolitian,30,1,15.0,afternoon,6.210138,medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45497,30.0,4.8,windy,high,1,meal,motorcycle,0.0,no,metropolitian,32,0,10.0,morning,1.489846,short
45498,21.0,4.6,windy,jam,0,buffet,motorcycle,1.0,no,metropolitian,36,0,15.0,evening,NaN,NaN
45499,30.0,4.9,cloudy,low,1,drinks,scooter,0.0,no,metropolitian,16,0,15.0,night,4.657195,short
45500,20.0,4.7,cloudy,high,0,snack,motorcycle,1.0,no,metropolitian,26,0,5.0,afternoon,6.232393,medium


In [9]:
temp_df=df.copy().dropna()

In [10]:
X=temp_df.drop(columns=['time_taken'])
y=temp_df['time_taken']

In [11]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [12]:
print("Training data size:", X_train.shape)
print("Testing data size:", X_test.shape)

Training data size: (30156, 15)
Testing data size: (7539, 15)


In [13]:
pt=PowerTransformer()
y_train_trans=pt.fit_transform(y_train.values.reshape(-1,1))
y_test_trans=pt.transform(y_test.values.reshape(-1,1))

In [14]:
num_cols = ["age","ratings","pickup_time_minutes","distance"]

nominal_cat_cols = ['weather',
                    'type_of_order',
                    'type_of_vehicle',
                    "festival",
                    "city_type",
                    "is_weekend",
                    "order_time_of_day"]

ordinal_cat_cols = ["traffic","distance_type"]

In [15]:
# generate order for ordinal encoding

traffic_order = ["low","medium","high","jam"]

distance_type_order = ["short","medium","long","very_long"]

In [16]:
# build a preprocessor

preprocessor = ColumnTransformer(transformers=[
    ("scale", MinMaxScaler(), num_cols),
    ("nominal_encode", OneHotEncoder(drop="first",handle_unknown="ignore",
                                     sparse_output=False), nominal_cat_cols),
    ("ordinal_encode", OrdinalEncoder(categories=[traffic_order,distance_type_order],
                                      encoded_missing_value=-999,
                                      handle_unknown="use_encoded_value",
                                      unknown_value=-1), ordinal_cat_cols)
],remainder="passthrough",n_jobs=-1,force_int_remainder_cols=False,verbose_feature_names_out=False)


preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('scale', ...), ('nominal_encode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",-1
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and `

In [17]:
X_train_trans=preprocessor.fit_transform(X_train)
X_test_trans=preprocessor.transform(X_test)

c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\compose\_column_transformer.py:978: FutureWarning: The parameter `force_int_remainder_cols` is deprecated and will be removed in 1.9. It has no effect. Leave it to its default value to avoid this warning.
  warnings.warn(


In [18]:
from sklearn.ensemble import RandomForestRegressor
import optuna
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.compose import TransformedTargetRegressor

In [19]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators",10,500),
            "max_depth": trial.suggest_int("max_depth",1,30),
            "max_features": trial.suggest_categorical("max_features",[None,"sqrt","log2"]),
            "min_samples_split": trial.suggest_int("min_samples_split",2,10),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf",1,10),
            "max_samples": trial.suggest_float("max_samples",0.5,1),
            "random_state": 42,
            "n_jobs": -1,
        }

        # log model parameters
        mlflow.log_params(params)

        # build the model
        rf = RandomForestRegressor(**params)
        model = TransformedTargetRegressor(regressor=rf,transformer=pt)

        # train the model
        model.fit(X_train_trans,y_train)

        # get the predictions
        y_pred_train = model.predict(X_train_trans)
        y_pred_test = model.predict(X_test_trans)


        # perform cross validation
        cv_score = cross_val_score(model,
                                X_train_trans,
                                y_train,
                                cv=5,
                                scoring="neg_mean_absolute_error",
                                n_jobs=-1)

        # mean score
        mean_score = -(cv_score.mean())

        # log avg cross val error
        mlflow.log_metric("cross_val_error",mean_score)

        return mean_score

In [20]:
# create optuna study
study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective,n_trials=20,n_jobs=-1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

    # train the model on best parameters
    best_rf = RandomForestRegressor(**study.best_params)

    best_rf.fit(X_train_trans,y_train_trans.values.ravel())

    # get the predictions
    y_pred_train = best_rf.predict(X_train_trans)
    y_pred_test = best_rf.predict(X_test_trans)

    # get the actual predictions values
    y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
    y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))


    # perform cross validation
    model = TransformedTargetRegressor(regressor=best_rf,
                                        transformer=pt)


    scores = cross_val_score(model,
                         X_train_trans,
                         y_train,
                         scoring="neg_mean_absolute_error",
                         cv=5,n_jobs=-1)

    # log metrics
    mlflow.log_metric("training_error",mean_absolute_error(y_train,y_pred_train_org))
    mlflow.log_metric("test_error",mean_absolute_error(y_test,y_pred_test_org))
    mlflow.log_metric("training_r2",r2_score(y_train,y_pred_train_org))
    mlflow.log_metric("test_r2",r2_score(y_test,y_pred_test_org))
    mlflow.log_metric("cross_val",- scores.mean())

    # log the best model
    mlflow.sklearn.log_model(best_rf,artifact_path="model")

[I 2026-03-31 13:00:57,656] A new study created in memory with name: no-name-d40d4a15-5efc-4c10-bb5f-5f30e26cd27a
  0%|          | 0/20 [00:00<?, ?it/s]

🏃 View run redolent-flea-894 at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/4/runs/f5916a7a883b4b298d247adb1acdb6c2
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/4


Best trial: 3. Best value: 4.31939:   5%|▌         | 1/20 [00:25<07:57, 25.12s/it]

[I 2026-03-31 13:01:23,788] Trial 3 finished with value: 4.319392137094858 and parameters: {'n_estimators': 40, 'max_depth': 5, 'max_features': None, 'min_samples_split': 3, 'min_samples_leaf': 9, 'max_samples': 0.6278514881242754}. Best is trial 3 with value: 4.319392137094858.
🏃 View run sincere-sow-412 at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/4/runs/5d71fc71138340a083d6d51c725a3468
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/4


Best trial: 4. Best value: 3.57006:  10%|█         | 2/20 [00:34<04:43, 15.76s/it]

...

[I 2026-03-31 13:12:13,699] Trial 18 finished with value: 3.0882609145957955 and parameters: {'n_estimators': 224, 'max_depth': 30, 'max_features': None, 'min_samples_split': 8, 'min_samples_leaf': 7, 'max_samples': 0.9848130861260387}. Best is trial 10 with value: 3.0759721464229353.
🏃 View run defiant-shark-911 at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/4/runs/d2f82e39cc794be999a8c0a1647e563c
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/4


Best trial: 10. Best value: 3.07597: 100%|██████████| 20/20 [11:18<00:00, 33.92s/it]


[I 2026-03-31 13:12:17,089] Trial 19 finished with value: 3.0883098016530837 and parameters: {'n_estimators': 238, 'max_depth': 29, 'max_features': None, 'min_samples_split': 8, 'min_samples_leaf': 7, 'max_samples': 0.9763134717336369}. Best is trial 10 with value: 3.0759721464229353.


2026/03/31 13:12:49 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/31 13:12:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/4/runs/419bc7ec401f4ef5b953062ef8f15de4
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/4


In [23]:
mlflow.set_experiment("Exp 4 - LGB - HP Tuning")

2026/03/31 13:15:59 INFO mlflow.tracking.fluent: Experiment with name 'Exp 4 - LGB - HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/e5d2c59da5bc4f79a897d5d7a85a9b9e', creation_time=1774943161325, experiment_id='5', last_update_time=1774943161325, lifecycle_stage='active', name='Exp 4 - LGB - HP Tuning', tags={}, workspace='default'>

In [24]:
from lightgbm import LGBMRegressor
import optuna

In [25]:
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.compose import TransformedTargetRegressor

In [26]:
def objective(trial):
    with mlflow.start_run(nested=True):
        params = {
            "n_estimators": trial.suggest_int("n_estimators",10,200),
            "max_depth": trial.suggest_int("max_depth",1,40),
            "learning_rate": trial.suggest_float("learning_rate",0.1,0.8),
            "subsample": trial.suggest_float("subsample",0.5,1),
            "min_child_weight": trial.suggest_int("min_child_weight",5,20),
            "min_split_gain": trial.suggest_float("min_split_gain",0,10),
            "reg_lambda": trial.suggest_float("reg_lambda",0,100),
            "random_state": 42,
            "n_jobs": -1,
        }

        # log model parameters
        mlflow.log_params(params)

        xgb_reg = LGBMRegressor(**params)
        model = TransformedTargetRegressor(regressor=xgb_reg,transformer=pt)

        # train the model
        model.fit(X_train_trans,y_train)

        # get the predictions
        y_pred_train = model.predict(X_train_trans)
        y_pred_test = model.predict(X_test_trans)


        # perform cross validation
        cv_score = cross_val_score(model,
                                X_train_trans,
                                y_train,
                                cv=5,
                                scoring="neg_mean_absolute_error",
                                n_jobs=-1)

        # mean score
        mean_score = -(cv_score.mean())
        # log avg cross val error
        mlflow.log_metric("cross_val_error",mean_score)

        return mean_score

In [28]:
# create optuna study
study = optuna.create_study(direction="minimize")

with mlflow.start_run(run_name="best_model"):
    # optimize the objective function
    study.optimize(objective,n_trials=50,n_jobs=-1,show_progress_bar=True)

    # log the best parameters
    mlflow.log_params(study.best_params)

    # log the best score
    mlflow.log_metric("best_score",study.best_value)

    # train the model on best parameters
    best_lgbm = LGBMRegressor(**study.best_params)

    best_lgbm.fit(X_train_trans,y_train_trans.values.ravel())

    # get the predictions
    y_pred_train = best_lgbm.predict(X_train_trans)
    y_pred_test = best_lgbm.predict(X_test_trans)

    # get the actual predictions values
    y_pred_train_org = pt.inverse_transform(y_pred_train.reshape(-1,1))
    y_pred_test_org = pt.inverse_transform(y_pred_test.reshape(-1,1))


    # perform cross validation
    model = TransformedTargetRegressor(regressor=best_lgbm,
                                        transformer=pt)


    scores = cross_val_score(model,
                         X_train_trans,
                         y_train,
                         scoring="neg_mean_absolute_error",
                         cv=5,n_jobs=-1)

    # log metrics
    mlflow.log_metric("training_error",mean_absolute_error(y_train,y_pred_train_org))
    mlflow.log_metric("test_error",mean_absolute_error(y_test,y_pred_test_org))
    mlflow.log_metric("training_r2",r2_score(y_train,y_pred_train_org))
    mlflow.log_metric("test_r2",r2_score(y_test,y_pred_test_org))
    mlflow.log_metric("cross_val",- scores.mean())

    # log the best model
    mlflow.sklearn.log_model(best_lgbm,artifact_path="model")

[I 2026-03-31 13:30:42,483] A new study created in memory with name: no-name-df4f251a-c2ad-45b6-a310-e8a087c2e66e


  0%|          | 0/50 [00:00<?, ?it/s]c:\Users\hp\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


🏃 View run thundering-foal-317 at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5/runs/1836078990bd421089611775a782be19
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5
🏃 View run wise-doe-665 at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5/runs/abf6888dbcf8481ab0ea1967487d9df4
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5
🏃 View run delightful-moth-653 at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5/runs/ce650041acf84aceb92559b8941edb9f
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5
🏃 View run rebellious-ray-504 at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5/runs/ab280563f0014a30884fe574f

Best trial: 3. Best value: 3.72164:   2%|▏         | 1/50 [00:22<18:27, 22.61s/it]

[I 2026-03-31 13:31:07,391] Trial 3 finished with value: 3.7216391811733027 and parameters: {'n_estimators': 16, 'max_depth': 11, 'learning_rate': 0.4069788586590587, 'subsample': 0.6306370125760903, 'min_child_weight': 15, 'min_split_gain': 6.999021818383686, 'reg_lambda': 43.46968480619784}. Best is trial 3 with value: 3.7216391811733027.


...

[I 2026-03-31 13:36:09,368] Trial 48 finished with value: 3.436328268688154 and parameters: {'n_estimators': 117, 'max_depth': 25, 'learning_rate': 0.5738333487151315, 'subsample': 0.9475222769639077, 'min_child_weight': 10, 'min_split_gain': 0.6753676735287026, 'reg_lambda': 35.05657158812735}. Best is trial 9 with value: 3.18351631442306.
🏃 View run monumental-dog-593 at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5/runs/f8164dd9680d4740aaef88636c462fd9
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5


Best trial: 9. Best value: 3.18352: 100%|██████████| 50/50 [05:26<00:00,  6.53s/it]


[I 2026-03-31 13:36:11,108] Trial 49 finished with value: 3.3405552797257982 and parameters: {'n_estimators': 117, 'max_depth': 25, 'learning_rate': 0.40453556771905885, 'subsample': 0.9430593719210822, 'min_child_weight': 10, 'min_split_gain': 0.6133405775169966, 'reg_lambda': 34.563613558540446}. Best is trial 9 with value: 3.18351631442306.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000875 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 352
[LightGBM] [Info] Number of data points in the train set: 30156, number of used features: 25
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
[LightGBM] [Warning] No further splits with positive gain, best gain:

2026/03/31 13:36:20 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/31 13:36:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run best_model at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5/runs/f9f8a215288748e99d3b6ed0068fa0e6
🧪 View experiment at: https://dagshub.com/rahulpatel16092005/swiggy-delivery-time-prediction.mlflow/#/experiments/5
